# Template Matching für den Chirp-Mass-Proxy (ET-HF-IMBH)

Bank wird einmal erzeugt und gespeichert, die Filterung läuft batchweise auf der GPU. Die Extrinsik wird **nicht** mehr aus der Injektion übernommen (blind). Die Logik steht in `chirp_mass_matcher.py`.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import numpy as np, matplotlib.pyplot as plt, torch
from dingo.core.posterior_models.build_model import build_model_from_kwargs
from dingo.gw.injection import Injection
import chirp_mass_matcher as cmm

MODEL_NAME = "stage1"   # anpassen
MODEL_PATH = f"/scratch/tpausch/models/{MODEL_NAME}/model_latest.pt"
BANK_PATH  = f"/scratch/tpausch/test_data/{MODEL_NAME}/chirp_mass_bank.pt"
INJECTION_DIR = f"/scratch/tpausch/test_data/{MODEL_NAME}/injections"

pm = build_model_from_kwargs(filename=MODEL_PATH, device="cpu", load_training_info=False)
generator = Injection.from_posterior_model_metadata(pm.metadata)
cfg = cmm.BankConfig.from_metadata(pm.metadata)   # f 6-256, dF 0.125, Mc 75-1200, m1/m2-Constraints
del pm
print(cfg)

## 1. Bank bauen (einmalig)
Gitter: Mc geometrisch (3 %), q ∈ 5 Werte, χ₁,χ₂ ∈ {−0.6, 0, 0.6}, θ_jn ∈ {0.6, π/2}. Alles in `BankConfig` änderbar. Mit `n_workers` parallelisieren.

In [ ]:
if os.path.exists(BANK_PATH):
    bank = cmm.TemplateBank.load(BANK_PATH, device="cuda")
else:
    h, grid = cmm.build_bank(cfg, cmm.make_dingo_h0(generator, cfg), n_workers=8)
    bank = cmm.TemplateBank(h, grid, cfg, device="cuda")
    bank.save(BANK_PATH)
print(len(bank), "Templates,", bank.n_mc, "Mc-Gitterpunkte")
matcher = cmm.ChirpMassMatcher(bank, oversample=4, t_window=0.125, chunk=256)

## 2. Sanity-Check (rauschfrei)
Fitting-Faktor sollte nahe 1 liegen, `mc_refined` nahe am wahren Wert. Schlägt das fehl, stimmt meist eine Konvention nicht (Array-Länge, `h_plus`-API, Zeitfenster).

In [ ]:
np.random.seed(0)
for _ in range(5):
    theta = {k: float(v) for k, v in generator.prior.sample().items()}
    cmm.sanity_check(matcher, generator, theta)

## 3. Injektionen auswerten

In [ ]:
import time
t0 = time.time()
res = cmm.evaluate_injections(matcher, INJECTION_DIR, max_injections=None, kernel_halfwidth=45.0)
print(f"{len(res)} Injektionen in {time.time()-t0:.1f} s")
res.to_csv(os.path.join(os.path.dirname(INJECTION_DIR), "matched_filter_results.csv"), index=False)
res[[c for c in res if c.startswith("inside_")]].mean()   # Anteil innerhalb Kernel +-45

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sc = ax[0].scatter(res.chirp_mass_true, res.mc_refined, c=res.snr_best, s=18, cmap="viridis")
lim = [75, 1200]; ax[0].plot(lim, lim, "k--"); plt.colorbar(sc, ax=ax[0], label="best network SNR")
ax[0].set(xlabel="true Mc", ylabel="template Mc (refined)")
ax[1].scatter(res.chirp_mass_true, res.err_mc_refined, s=18, c=res.snr_best)
ax[1].axhspan(-45, 45, color="g", alpha=0.15, label="Kernel +-45")
ax[1].axhline(0, color="k", ls="--"); ax[1].set(xlabel="true Mc", ylabel="error [Msun]"); ax[1].legend()
for a in ax: a.grid(alpha=0.3)
plt.show()

# Fehlerquantile in Mc-Bins -> Grundlage fuer die Kernel-Breite
bins = np.geomspace(75, 1200, 8)
res["bin"] = np.digitize(res.chirp_mass_true, bins)
print(res.groupby("bin").err_mc_refined.agg(lambda x: np.quantile(np.abs(x), [0.5, 0.9, 0.99]).round(1).tolist()))